# Wine Style Clustering — 00: Preprocessing

**Input:** `data/raw/wines_raw.csv`  
**Output:** `data/raw/wines_clean_llm.csv` + `data/processed/wines_model.csv`

Steps:
1. Basic cleaning — drop nulls, duplicates, strip HTML
2. Merge LLM-cleaned descriptions
3. Normalise grape varieties
4. Save modelling subset

---

## 0. Setup

In [8]:
import os
os.chdir(os.path.expanduser('~/Documents/wine-style-clustering'))

import re
import pandas as pd

RAW_PATH    = 'data/raw/wines_raw.csv'
CLEAN_PATH  = 'data/raw/wines_clean.csv'
LLM_PATH    = 'data/raw/wines_clean_llm.csv'
MODEL_PATH  = 'data/processed/wines_model.csv'

print('Ready.')


Ready.


## 1. Basic cleaning

In [9]:
df = pd.read_csv(RAW_PATH)
print(f'Loaded: {len(df):,} rows')

df = df.dropna(subset=['description'])
df['score'] = df['score'].replace(0.0, float('nan'))
df = df.drop_duplicates(subset=['description'], keep='first')
print(f'After dedup: {len(df):,}')

# Strip HTML tags
df['description'] = df['description'].apply(lambda t: re.sub(r'<[^>]+>', '', str(t)).strip())

if 'Brand' in df.columns:
    df = df.drop(columns=['Brand'])

# Fill metadata nulls
meta_cols = [c for c in df.columns if c not in ('description', 'score', 'url')]
df[meta_cols] = df[meta_cols].fillna('Unknown')

# Drop wines with unknown colour and fix Body artifact
df = df[df['Colour'] != 'Unknown'].reset_index(drop=True)
df.loc[df['Body'] == 'Yes', 'Body'] = 'Unknown'

df.to_csv(CLEAN_PATH, index=False)
print(f'Saved → {CLEAN_PATH}')
print(df['Colour'].value_counts())


Loaded: 23,374 rows
After dedup: 23,281
Saved → data/raw/wines_clean.csv
Colour
White     10426
Red        9287
Rosé       3398
Orange      155
Name: count, dtype: int64


## 2. Merge LLM-cleaned descriptions

Run `python src/clean_descriptions.py` first to generate cleaned descriptions.  
Claude Haiku removes winemaking details, provenance and food pairing — leaving pure sensory language.

In [10]:
df = pd.read_csv(CLEAN_PATH)

try:
    old = pd.read_csv(LLM_PATH)[['url', 'description_clean']]
    df  = df.merge(old, on='url', how='left')
    n_done = df['description_clean'].notna().sum()
    print(f'Merged: {n_done:,} cleaned | {df["description_clean"].isna().sum():,} still raw')
except FileNotFoundError:
    df['description_clean'] = float('nan')
    print('No LLM file found — run src/clean_descriptions.py first')

df.to_csv(LLM_PATH, index=False)
print(f'Saved → {LLM_PATH}')


Merged: 23,266 cleaned | 0 still raw
Saved → data/raw/wines_clean_llm.csv


## 3. Normalise grape varieties

Maps raw `Grapes` field to `grape_normalized`.  
Named blends identified by variety composition, then fallback to dominant variety or colour-based label.

In [11]:
BORDEAUX_GRAPES      = {'cabernet sauvignon', 'merlot', 'cabernet franc', 'petit verdot', 'malbec'}
GSM_GRAPES           = {'grenache', 'syrah', 'mourvèdre', 'mourvedre', 'garnacha', 'shiraz'}
SUPER_TUSCAN         = {'sangiovese', 'cabernet sauvignon', 'cabernet franc', 'merlot'}
CHIANTI_GRAPES       = {'sangiovese', 'canaiolo', 'colorino', 'trebbiano', 'malvasia'}
CHAMPAGNE_GRAPES     = {'chardonnay', 'pinot noir', 'pinot meunier'}
PROSECCO_GRAPES      = {'glera', 'pinot noir', 'chardonnay'}
CAVA_GRAPES          = {'macabeo', 'parellada', 'xarel·lo', 'xarel-lo', 'chardonnay', 'pinot noir'}
BORDEAUX_WHITE       = {'sauvignon blanc', 'sauvignon gris', 'sémillon', 'semillon', 'muscadelle'}
RHONE_WHITE          = {'marsanne', 'roussanne', 'viognier', 'grenache blanc', 'clairette'}
SOUTHERN_RHONE_WHITE = {'bourboulenc', 'clairette', 'roussanne', 'marsanne', 'viognier',
                        'grenache blanc', 'rolle', 'ugni blanc'}
NORTHERN_RHONE       = {'syrah', 'shiraz', 'viognier'}
PROVENCE_GRAPES      = {'cinsault', 'grenache', 'garnacha', 'mourvèdre', 'mourvedre',
                        'rolle', 'syrah', 'shiraz', 'carignan', 'clairette'}
LANGUEDOC_GRAPES     = {'grenache', 'garnacha', 'carignan', 'cariñena', 'syrah', 'shiraz',
                        'mourvèdre', 'mourvedre', 'cinsault', 'rolle', 'viura', 'tibouren',
                        'vermentino', 'grenache blanc', 'garnacha blanca'}
RIOJA_GRAPES         = {'tempranillo', 'tinto fino', 'graciano', 'garnacha', 'grenache', 'viura', 'carignan'}
VALPOLICELLA         = {'corvina', 'corvinone', 'rondinella', 'molinara', 'oseleta'}
DOURO_GRAPES         = {'touriga nacional', 'touriga franca', 'tinta roriz', 'tinta barroca',
                        'tinto cão', 'sousão', 'alfrocheiro', 'tinta amarela'}
TOKAJ_GRAPES         = {'furmint', 'hárslevelu', 'harslevelu', 'sárga muskotály', 'zéta',
                        'yellow muscat', 'muscat blanc'}
ETNA_GRAPES          = {'nerello mascalese', 'nerello cappuccio', 'carricante', 'catarratto', 'minnella'}
SOAVE_GRAPES         = {'garganega', 'trebbiano di soave', 'trebbiano', 'chardonnay'}


def extract_grape_set(val):
    val = re.sub(r'\d+\.?\d*\s*%\s*', '', val.replace('\n', ', '))
    return {p.split('/')[0].strip().lower() for p in re.split(r'[,]', val) if p.strip()}


def get_dominant_grape(val):
    parts = [p.strip() for p in re.split(r'[,]', val.replace('\n', ', ')) if p.strip()]
    pcts  = [float(m.group(1)) if (m := re.search(r'(\d+\.?\d*)\s*%', p)) else None for p in parts]
    # All 0% — return first grape
    if pcts and all(p == 0.0 for p in pcts if p is not None):
        first = re.sub(r'\d+\.?\d*\s*%\s*', '', parts[0]).split('/')[0].strip().lower()
        return first or None
    # One grape >50%
    for part, pct in zip(parts, pcts):
        if pct and pct > 50:
            name = re.sub(r'\d+\.?\d*\s*%\s*', '', part).split('/')[0].strip().lower()
            return name or None
    return None


def normalize_grape(row):
    val = row['Grapes']
    wine_type = str(row.get('Wine Type', '')).strip()
    if pd.isna(val) or str(val).strip().lower() == 'unknown':
        return None
    val = str(val)
    parts = [p.strip() for p in re.split(r'[,]',
              re.sub(r'\d+\.?\d*\s*%\s*', '', val.replace('\n', ', '))) if p.strip()]
    if not parts:
        return None
    # Single variety
    if len(parts) == 1:
        return parts[0].split('/')[0].strip().lower() or None

    g = extract_grape_set(val)

    if wine_type == 'Sparkling':
        if 'glera' in g and g <= PROSECCO_GRAPES:                             return 'prosecco blend'
        if g <= CHAMPAGNE_GRAPES:                                              return 'champagne blend'
        if ('macabeo' in g or 'xarel·lo' in g) and g <= CAVA_GRAPES:          return 'cava blend'
        return 'sparkling blend'

    if len(g) >= 2 and g <= BORDEAUX_GRAPES:                                  return 'bordeaux blend'
    if 'sauvignon blanc' in g and g <= BORDEAUX_WHITE:                        return 'bordeaux white blend'
    if 'sangiovese' in g and g <= SUPER_TUSCAN:                               return 'super tuscan blend'
    if 'sangiovese' in g and g <= CHIANTI_GRAPES:                             return 'chianti blend'

    if 'grenache' in g or 'garnacha' in g:
        if ('syrah' in g or 'shiraz' in g) and ('mourvèdre' in g or 'mourvedre' in g) and g <= GSM_GRAPES:
            return 'gsm blend'
        if ('syrah' in g or 'shiraz' in g) and g <= GSM_GRAPES:               return 'rhone blend'
        if ('mourvèdre' in g or 'mourvedre' in g) and 'syrah' not in g:       return 'rhone blend'
        if ('carignan' in g or 'cariñena' in g) and g <= LANGUEDOC_GRAPES:    return 'languedoc blend'

    if ('syrah' in g or 'shiraz' in g) and 'viognier' in g and g <= NORTHERN_RHONE: return 'northern rhone blend'
    if 'cinsault' in g and g <= PROVENCE_GRAPES:                              return 'provence blend'
    if ('marsanne' in g or 'roussanne' in g) and g <= RHONE_WHITE:            return 'white rhone blend'
    if ('bourboulenc' in g or 'clairette' in g) and g <= SOUTHERN_RHONE_WHITE: return 'southern rhone white blend'
    if 'garganega' in g and g <= SOAVE_GRAPES:                                return 'soave blend'
    if 'corvina' in g and g <= VALPOLICELLA:                                  return 'valpolicella blend'
    if ('touriga nacional' in g or 'touriga franca' in g) and g <= DOURO_GRAPES: return 'douro blend'
    if 'furmint' in g and g <= TOKAJ_GRAPES:                                  return 'tokaj blend'
    if ('tempranillo' in g or 'tinto fino' in g) and g <= RIOJA_GRAPES:       return 'rioja blend'
    if ('nerello mascalese' in g or 'carricante' in g) and g <= ETNA_GRAPES:  return 'etna blend'

    dominant = get_dominant_grape(val)
    if dominant:
        return f'{dominant} blend'

    if wine_type == 'Fortified': return 'fortified blend'
    return {'Red': 'red blend', 'White': 'white blend',
            'Rosé': 'rosé blend', 'Orange': 'orange blend'}.get(str(row.get('Colour','')).strip(), 'blend')


df = pd.read_csv(LLM_PATH)
df['grape_normalized'] = df.apply(normalize_grape, axis=1)
df.to_csv(LLM_PATH, index=False)

print('grape_normalized — top 50:')
print(df['grape_normalized'].value_counts().head(50))
print(f'\nUnique: {df["grape_normalized"].nunique()} | Null: {df["grape_normalized"].isna().sum()} | blend: {(df["grape_normalized"]=="blend").sum()}')
print(f'Saved → {LLM_PATH}')


grape_normalized — top 50:
grape_normalized
chardonnay                    2760
pinot noir                    2226
bordeaux blend                1325
champagne blend               1074
sauvignon blanc                728
riesling                       570
cabernet sauvignon             539
provence blend                 515
syrah                          488
rhone blend                    418
sangiovese                     393
grenache                       373
nebbiolo                       360
bordeaux white blend           343
sparkling blend                326
tempranillo                    315
white rhone blend              278
cabernet franc                 265
gsm blend                      259
glera                          244
merlot                         232
white blend                    218
valpolicella blend             182
chenin blanc                   178
languedoc blend                178
malbec                         174
grenache blend                 174
viognier   

In [15]:
OAK_MAP = {
    'Oaked':         'oaked',
    'Lightly Oaked': 'oaked',
    'Yes':           'oaked',
    'Oak':           'oaked',
    'Unoaked':       'unoaked',
    'No':            'unoaked',
    'Unknown':       'unknown',
}

def normalize_oak(val):
    val = str(val).strip()
    if val in OAK_MAP:
        return OAK_MAP[val]
    if 'oak' in val.lower():
        return 'oaked'
    return 'unknown'

df['oak_normalized'] = df['Oak'].apply(normalize_oak)
print(df['oak_normalized'].value_counts())
df.to_csv(LLM_PATH, index=False)

oak_normalized
unknown    10914
oaked       8812
unoaked     3540
Name: count, dtype: int64


## 4. Save modelling subset

Filter to Still/Unknown wine types, Red + White + Rosé, minimum 15 words.

In [16]:
df = pd.read_csv(LLM_PATH)
df['desc_len'] = df['description_clean'].fillna(df['description']).str.split().str.len()

df_model = df[
    df['Wine Type'].isin(['Still', 'Unknown']) &
    df['Colour'].isin(['Red', 'White', 'Rosé']) &
    (df['desc_len'] >= 15)
].reset_index(drop=True)

df_model.to_csv(MODEL_PATH, index=False)

print(f'Modelling subset saved → {MODEL_PATH}')
print(f'  Total:  {len(df_model):,}')
print(f'  Red:    {(df_model["Colour"]=="Red").sum():,}')
print(f'  White:  {(df_model["Colour"]=="White").sum():,}')
print(f'  Rosé:   {(df_model["Colour"]=="Rosé").sum():,}')


Modelling subset saved → data/processed/wines_model.csv
  Total:  19,241
  Red:    8,783
  White:  8,364
  Rosé:   2,094
